<a href="https://colab.research.google.com/github/raki-rankawat/stm32-mobilenet/blob/main/CIFAR10_Model_Comparision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# =====================================================
# Imports
# =====================================================

import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# =====================================================
# Data Loaders
# =====================================================

batch_size = 64

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

train_data = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

test_data = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

100%|██████████| 170M/170M [00:03<00:00, 46.3MB/s]


In [4]:
# =====================================================
# MobileNetV2 Components
# =====================================================

class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()

        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        self.block = nn.Sequential(
            # 1x1 Expansion
            nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            # 3x3 Depthwise Convolution
            nn.Conv2d(hidden_dim,
                      hidden_dim,
                      3,
                      stride=stride,
                      padding=1,
                      groups=hidden_dim,
                      bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            # 1x1 Projection
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        else:
            return self.block(x)

In [5]:
# =====================================================
# MobileNetV2 Model (CIFAR10 Version)
# =====================================================

class MobileNetV2_CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Initial Conv Layer
        self.initial = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        # Inverted Residual Blocks
        self.features = nn.Sequential(
            InvertedResidual(32, 16, stride=1, expand_ratio=1),

            InvertedResidual(16, 24, stride=1, expand_ratio=6),
            InvertedResidual(24, 24, stride=1, expand_ratio=6),

            InvertedResidual(24, 32, stride=2, expand_ratio=6),
            InvertedResidual(32, 32, stride=1, expand_ratio=6),
            InvertedResidual(32, 32, stride=1, expand_ratio=6),

            InvertedResidual(32, 64, stride=2, expand_ratio=6),
            InvertedResidual(64, 64, stride=1, expand_ratio=6),
        )

        # Final Convolution
        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 1280, 1, bias=False),
            nn.BatchNorm2d(1280),
            nn.ReLU6(inplace=True)
        )

        # Classifier
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.initial(x)
        x = self.features(x)
        x = self.final_conv(x)

        # Global Average Pooling
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)

        x = self.classifier(x)
        return x

In [6]:
# =====================================================
# VGG-Style CNN
# =====================================================

class VGG_CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 → 16

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 → 8

            # Block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 8 → 4
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [16]:
# =====================================================
# Device, Seed, Model Initialization
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

mobilenet = MobileNetV2_CIFAR().to(device)
mobilenet.load_state_dict(torch.load("/content/drive/My Drive/Colab Notebooks/cifar10_mobilenetv2_model.pth", map_location=torch.device('cpu')))

vgg_model = VGG_CIFAR().to(device)
vgg_model.load_state_dict(torch.load("/content/drive/My Drive/Colab Notebooks/cifar10_vgg_model.pth", map_location=torch.device('cpu')))

<All keys matched successfully>

In [9]:
# =====================================================
# Parameter Count
# =====================================================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("MobileNet Params:", count_parameters(mobilenet))
print("VGG Params:", count_parameters(vgg_model))

MobileNet Params: 229194
VGG Params: 3249994


In [11]:
# =====================================================
# Inference Latency
# =====================================================

def measure_latency(model, runs=100):
    model.eval()
    dummy = torch.randn(1,3,32,32).to(device)

    for _ in range(10):  # warm-up
        _ = model(dummy)

    torch.cuda.synchronize() if device.type == 'cuda' else None

    start = time.time()
    for _ in range(runs):
        _ = model(dummy)
    torch.cuda.synchronize() if device.type == 'cuda' else None
    end = time.time()

    return (end-start)/runs

print("MobileNet Latency:", measure_latency(mobilenet))
print("VGG Latency:", measure_latency(vgg_model))

MobileNet Latency: 0.003608386516571045
VGG Latency: 0.0011087107658386232


In [14]:
# =====================================================
# Model Size
# =====================================================

torch.save(mobilenet.state_dict(), "mobilenet.pth")
torch.save(vgg_model.state_dict(), "vgg.pth")

print("MobileNet Size (MB):", os.path.getsize("mobilenet.pth")/1e6)
print("VGG Size (MB):", os.path.getsize("vgg.pth")/1e6)

MobileNet Size (MB): 1.005387
VGG Size (MB): 13.020041


In [17]:
# =====================================================
# Accuracy Comparison
# =====================================================

def test(model, loader, criterion):
    model.eval()
    total, correct, running_loss = 0, 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)

            batch = y.size(0)
            running_loss += loss.item() * batch
            preds = outputs.argmax(1)
            correct += (preds == y).sum().item()
            total += batch

    return running_loss/total, correct/total

In [18]:
mobilenet_loss, mobilenet_acc = test(mobilenet, test_loader, criterion)
vgg_loss, vgg_acc = test(vgg_model, test_loader, criterion)

print("MobileNet Accuracy:", mobilenet_acc * 100)
print("VGG Accuracy:", vgg_acc * 100)

MobileNet Accuracy: 85.49
VGG Accuracy: 88.71
